# 03 - Khana: extract, split, cap (Day 2-3, not blocking Day 1)
Edit `ARCHIVE` to your real archive filename on Drive. Extraction takes a while and must be repeated every new Colab session.
Do not save this notebook back to GitHub.

In [ ]:
GITHUB_USER = "YOUR-GITHUB-USERNAME"   # <-- EDIT THIS (the account that owns the AaharAI repo)

from google.colab import drive
drive.mount("/content/drive")
!pip -q install rapidfuzz
import sys
!git -C /content/AaharAI pull || git clone https://github.com/{GITHUB_USER}/AaharAI.git /content/AaharAI
sys.path.insert(0, "/content/AaharAI")
%load_ext autoreload
%autoreload 2

In [ ]:
import os, tarfile
ARCHIVE = "/content/drive/MyDrive/AaharAI_Data/<your-archive-file>.gz"   # <-- EDIT THIS
if not os.path.exists("/content/khana/khana"):
    os.makedirs("/content/khana", exist_ok=True)
    with tarfile.open(ARCHIVE, "r:gz") as t:
        try:
            t.extractall("/content/khana", filter="data")
        except TypeError:
            t.extractall("/content/khana")
print("classes:", len([d for d in os.listdir("/content/khana/khana") if not d.startswith(".")]))

In [ ]:
import hashlib, pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split
from src.config import DATA_DIR

ROOT, SEED, CAP = Path("/content/khana/khana"), 42, 1000
rows = [(f"{d.name}/{f.name}", d.name)
        for d in sorted(ROOT.iterdir()) if d.is_dir() and not d.name.startswith(".")
        for f in d.iterdir() if f.is_file() and not f.name.startswith(".")]
df = pd.DataFrame(rows, columns=["path", "label"])
print(len(df), "files (expect 131,819)")

def md5(rel):
    h = hashlib.md5()
    with open(ROOT / rel, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

with ThreadPoolExecutor(8) as ex:
    df["md5"] = list(ex.map(md5, df.path))
dups = df.duplicated("md5", keep="first")
print("exact duplicates removed:", int(dups.sum()))
df = df[~dups].reset_index(drop=True)

train, rest = train_test_split(df, test_size=0.30, stratify=df.label, random_state=SEED)
val, test = train_test_split(rest, test_size=0.50, stratify=rest.label, random_state=SEED)
train = train.sample(frac=1, random_state=SEED)
train = train[train.groupby("label").cumcount() < CAP]          # cap the TRAINING split only

splits = pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")])
splits[["path", "label", "split"]].to_csv(DATA_DIR / "khana_splits.csv", index=False)
print(splits.split.value_counts().to_string())
print("smallest / largest train class:", train.label.value_counts().min(), "/", train.label.value_counts().max())